# Project 01 (basic) — Sentiment classification with an LSTM

**Module 09 — NLP 2** · Format: **Jupyter notebook** (`sentiment_lstm.ipynb`)

In module 08 (NLP 1) you modelled texts with *counts*: n-grams, TF-IDF, naive
Bayes. Those models see a word as an atomic symbol and word order only through
short n-gram windows. **Neural** models do two things differently:

1. **Embeddings** — every word becomes a *dense, learned* vector (instead of a
   one-hot symbol). Similar words get similar vectors.
2. **Recurrence (RNN/LSTM)** — the model reads the sentence *word by word* and
   maintains a hidden state that carries context across the whole sequence.

Here you build an **LSTM** (long short-term memory) that classifies movie reviews
and product reviews as *positive* or *negative*, and you compare it honestly
against a bag-of-words baseline.

> **Plenty of instruction:** data preparation, vocabulary, batching and the
> training scaffold are given. Your three tasks hit exactly the learning points:
> assemble the LSTM model, write a training step, and evaluate/compare.


## What you learn

- How to prepare text for a neural model: **vocabulary**, **integer encoding**,
  **padding** and **packing** of variable sequence lengths.
- How **`nn.Embedding` + `nn.LSTM` + a classification head** interact in PyTorch.
- Why one uses `pack_padded_sequence` on padded batches (the model should *not*
  learn from the padding).
- What a **training loop** looks like (forward -> loss -> `backward` -> `step`).
- That an LSTM does not automatically beat a strong BoW baseline on *short*
  sentences — and *why* (an outlook on attention/transformers in projects 02 & 03).

## Prior knowledge

- Script parts 1-3 (embeddings, RNN/LSTM, vanishing gradient).
- PyTorch basics from **module 05** (`nn.Module`, optimizer, `loss.backward()`).


## Setup

Requires `torch`, `scikit-learn`, `numpy` (all in the repo `requirements.txt`). The
first code cell **downloads the dataset** (UCI *Sentiment Labelled Sentences*,
~85 KB) into `datasets/` and caches it (kept out of the repository by
`.gitignore`).

```bash
source ../../../../.venv/bin/activate
jupyter lab      # or open sentiment_lstm.ipynb in VS Code, kernel = repo .venv
```

Training runs in **a few minutes on the CPU**. If you have an Apple GPU (MPS) or
CUDA, the notebook uses it automatically.


## Part A — Loading & preparing the data *(given)*

The dataset contains **3000 short sentences** from real reviews on **IMDb, Amazon
and Yelp**, half of them positive (label 1) and half negative (label 0). Example:

> *"A very, very, very slow-moving, aimless movie ..."* -> 0
> *"Wow... Loved this place."* -> 1

We load, shuffle (fixed seed) and split into train/test.


In [ ]:
# ---- Load the dataset (real reviews: IMDb + Amazon + Yelp) ----------------
import os, io, zipfile, urllib.request, random

DATA_DIR = "datasets"
os.makedirs(DATA_DIR, exist_ok=True)
FILES = ["imdb_labelled.txt", "amazon_cells_labelled.txt", "yelp_labelled.txt"]
URL = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"

if not all(os.path.exists(os.path.join(DATA_DIR, f)) for f in FILES):
    print("Downloading dataset ...")
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    raw = urllib.request.urlopen(req, timeout=60).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        for name in z.namelist():
            base = os.path.basename(name)
            if base in FILES:
                with z.open(name) as src, open(os.path.join(DATA_DIR, base), "wb") as dst:
                    dst.write(src.read())
    print("Done.")
else:
    print("Dataset already present.")

# Read the lines: every line is "sentence \t label"
examples = []  # list of (text, label)
for f in FILES:
    with open(os.path.join(DATA_DIR, f), encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line or "\t" not in line:
                continue
            text, label = line.rsplit("\t", 1)
            examples.append((text, int(label)))

print(f"{len(examples)} examples loaded.")
print("Positive example:", next(t for t, y in examples if y == 1)[:60])
print("Negative example:", next(t for t, y in examples if y == 0)[:60])

In [ ]:
# ---- Shuffle & train/test split (fixed seed = reproducible) ---------------
random.seed(42)
random.shuffle(examples)

n_test = 600
test_data  = examples[:n_test]
train_data = examples[n_test:]
print(f"Train: {len(train_data)}   Test: {len(test_data)}")

# Check the class balance
import numpy as np
tr_y = np.array([y for _, y in train_data])
print(f"Share of positives in train: {tr_y.mean():.2f}")

In [ ]:
# ---- Tokenization & vocabulary --------------------------------------------
import re
from collections import Counter

def tokenize(text):
    # Lowercase + words/numbers as tokens (punctuation is dropped)
    return re.findall(r"[a-z0-9']+", text.lower())

# Build the vocabulary from the TRAINING data only (no peeking at test!)
counter = Counter(tok for text, _ in train_data for tok in tokenize(text))
MIN_FREQ = 2  # rare words (frequency 1) -> <unk>, keeps the vocabulary small

PAD, UNK = "<pad>", "<unk>"
itos = [PAD, UNK] + [w for w, c in counter.most_common() if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
PAD_IDX, UNK_IDX = stoi[PAD], stoi[UNK]
VOCAB_SIZE = len(itos)
print(f"Vocabulary size: {VOCAB_SIZE}")

def encode(text):
    """Sentence -> list of integer IDs (unknown words -> <unk>)."""
    return [stoi.get(tok, UNK_IDX) for tok in tokenize(text)]

print("Encoding example:", encode("loved this place")[:10])

In [ ]:
# ---- PyTorch dataset & DataLoader with padding ----------------------------
import torch
from torch.utils.data import Dataset, DataLoader

device = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("Device:", device)

class SentimentDataset(Dataset):
    def __init__(self, data):
        self.data = [(encode(t), y) for t, y in data if len(encode(t)) > 0]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        ids, y = self.data[i]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(y, dtype=torch.float)

def collate(batch):
    """Pad a batch to a common length and return the true lengths as well.
    We need the lengths later for pack_padded_sequence."""
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    maxlen = lengths.max().item()
    padded = torch.full((len(seqs), maxlen), PAD_IDX, dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :len(s)] = s
    return padded, lengths, torch.stack(labels)

BATCH = 32
train_loader = DataLoader(SentimentDataset(train_data), batch_size=BATCH,
                          shuffle=True, collate_fn=collate)
test_loader  = DataLoader(SentimentDataset(test_data), batch_size=BATCH,
                          shuffle=False, collate_fn=collate)

xb, lb, yb = next(iter(train_loader))
print("Batch shape:", xb.shape, "lengths[:5]:", lb[:5].tolist())

### Task 1 — Assemble the LSTM model

Build the classifier. The architecture:

$$\text{IDs} \;\xrightarrow{\text{embedding}}\; \mathbf{e}_1\dots\mathbf{e}_T
\;\xrightarrow{\text{LSTM}}\; \mathbf{h}_1\dots\mathbf{h}_T \;\longrightarrow\;
\mathbf{h}_T \xrightarrow{\text{linear}} \text{logit}$$

To do in `forward` (three places marked with `# TODO`):

1. **Embedding:** `emb = self.embedding(x)` -> shape `(batch, T, embed_dim)`.
2. **Packing + LSTM:** we want the LSTM to **ignore the padding**. To that end,
   wrap the embeddings with `pack_padded_sequence(emb, lengths, batch_first=True,
   enforce_sorted=False)` and send them through `self.lstm`. The LSTM returns
   `(output, (h_n, c_n))` — we need **`h_n`**, the last hidden state (shape
   `(1, batch, hidden_dim)`).
3. **Head:** take the batch vector `h_n[-1]` (shape `(batch, hidden_dim)`) from
   `h_n`, apply dropout and map it through `self.fc` onto **one** logit per
   example; remove the last dimension with `.squeeze(1)`.

> Why `h_n` and not `output`? `h_n` is the state *after the last real word* of
> every sequence — exactly the summary of the whole sentence that we classify.


In [ ]:
# ---- TASK 1: LSTM classifier ----------------------------------------------
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        # 1) TODO: embedding lookup -> emb, shape (B, T, E)
        emb = ...
        # 2) TODO: wrap emb with pack_padded_sequence (batch_first=True,
        #    enforce_sorted=False, lengths.cpu()), send it through self.lstm,
        #    and pull out h_n  ->  _, (h_n, _) = self.lstm(packed)
        ...
        # 3) TODO: head: h_n[-1] (B, H) -> dropout -> self.fc -> .squeeze(1)
        raise NotImplementedError("Task 1: implement forward")

model = LSTMClassifier(VOCAB_SIZE, pad_idx=PAD_IDX).to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

### Task 2 — One training step

The loop structure (epochs, batches, evaluation) is given. Fill in the **core of a
training step** in `train_one_epoch` (four `# TODO` lines):

1. `optimizer.zero_grad()` — clear the old gradients.
2. **Forward:** `logits = model(xb, lengths)`.
3. **Loss:** `loss = criterion(logits, yb)` (binary cross entropy on the logits).
4. **Backward + update:** `loss.backward()`, then `optimizer.step()`.

We use `BCEWithLogitsLoss` — it combines sigmoid + binary cross entropy in a
numerically stable way, which is why the model outputs raw **logits** (not a
probability).


In [ ]:
# ---- Evaluation helper (given) --------------------------------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for xb, lengths, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb, lengths)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    return correct / total

In [ ]:
# ---- TASK 2: training step + training run ---------------------------------
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for xb, lengths, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        # TODO 1: optimizer.zero_grad()
        # TODO 2: logits = model(xb, lengths)
        # TODO 3: loss = criterion(logits, yb)
        # TODO 4: loss.backward()  and  optimizer.step()
        raise NotImplementedError("Task 2: implement the training step")
        total_loss += loss.item() * yb.size(0)
    return total_loss / len(loader.dataset)

EPOCHS = 8
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader)
    tr_acc = evaluate(model, train_loader)
    te_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch:2d} | loss {loss:.3f} | train acc {tr_acc:.3f} | test acc {te_acc:.3f}")

### Task 3 — Baseline comparison & your own sentences

**(a)** The bag-of-words baseline (`CountVectorizer` + `LogisticRegression` from
scikit-learn, as in module 08) is given. Note in the markdown reflection below how
it fares against the LSTM.

**(b)** Fill in `predict_sentiment` (two `# TODO`): encode a single sentence, send
it through the model as a batch of size 1, apply the sigmoid and return the
probability. Test some sentences of your own.


In [ ]:
# ---- (a) Bag-of-words baseline (given, for comparison) --------------------
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

vec = CountVectorizer(tokenizer=tokenize, token_pattern=None)
Xtr = vec.fit_transform([t for t, _ in train_data])
Xte = vec.transform([t for t, _ in test_data])
ytr = [y for _, y in train_data]
yte = [y for _, y in test_data]

clf = LogisticRegression(max_iter=1000)
clf.fit(Xtr, ytr)
bow_acc = clf.score(Xte, yte)
print(f"BoW + LogReg test acc: {bow_acc:.3f}")
print(f"LSTM          test acc: {evaluate(model, test_loader):.3f}")

In [ ]:
# ---- (b) TASK 3: classify your own sentences ------------------------------
@torch.no_grad()
def predict_sentiment(text):
    model.eval()
    ids = encode(text)
    if not ids:
        return 0.5
    # TODO 1: build a batch of size 1:
    #   x = torch.tensor([ids], ...).to(device);  lengths = torch.tensor([len(ids)])
    # TODO 2: logit = model(x, lengths);  prob = torch.sigmoid(logit).item();  return prob
    raise NotImplementedError("Task 3b: implement predict_sentiment")

for s in ["I absolutely loved this movie, fantastic!",
          "Terrible product, broke after one day.",
          "The food was okay, nothing special."]:
    p = predict_sentiment(s)
    print(f"[{'POS' if p>0.5 else 'NEG'} p={p:.2f}]  {s}")

## Reflection (short, written)

Answer in your own words:

1. **LSTM vs. BoW — the aha moment:** how large is the gap? Which model wins, and
   what does that tell you about the relationship between model capacity and the
   amount of data?

2. **Padding & packing:** what would happen if we did *not* hide the padding with
   `pack_padded_sequence` but simply let the `<pad>` tokens run through the LSTM?

3. **Overfitting:** train accuracy climbs towards 1.0, test accuracy stays below.
   How do you recognize overfitting, and which two knobs (visible in the code) work
   against it?

4. **Outlook:** the LSTM reads strictly *sequentially* and has to squeeze the whole
   sentence into **one** vector $\mathbf{h}_T$. What problem does that create for
   long sequences — and how does **attention** (project 02) attack it?

> Write down your answers first, then compare them with the **reference answers at
> the end of the solution** in the folder `solution/`.

> **Reference numbers** (fixed seed): LSTM test accuracy ~0.78-0.80 (train accuracy
> ~0.99 -> clear overfitting), BoW+LogReg ~0.85-0.86. The exact numbers vary
> slightly with the hardware — the fact that the BoW baseline *wins* here is the
> actual learning point.
